In [ ]:
"""Import and load everything"""

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

depression_file = pd.read_csv("C:/Users/treyk/Downloads/student_depression_dataset.csv")

In [ ]:
"""Clean data""" 
#I feel like im doing something wrong in my cleaning because the data is only plotting if i call depression_file rather than _cleaned which i assume is what that means.
# Which i also assume means im currently using the raw uncleaned data for the plots so they're obviously wrong 
# Pretty sure the issue is im not converting the strings properly, tried asking chatgpt for help on this block and im still a lil lost:/

depression_file_cleaned = depression_file.copy() 

for column in depression_file_cleaned.select_dtypes(include='object').columns:
    depression_file_cleaned[column] = depression_file_cleaned[column].str.replace("'", "").str.strip()

print("\nMissing Values per Column:")
print(depression_file_cleaned.isnull().sum())

binary_columns = ['Have you ever had suicidal thoughts ?', 'Family History of Mental Illness', 'Depression']
binary_map = {'Yes': 1, 'No': 0}
for column in binary_columns:
    depression_file_cleaned[column] = depression_file_cleaned[column].astype(str).str.strip().str.lower().map({'yes': 1, 'no': 0})

numeric_conversion = ['Financial Stress', 'Academic Pressure', 'Work Pressure', 
                      'Study Satisfaction', 'Job Satisfaction', 'CGPA', 'Work/Study Hours', 'Age', 'Sleep Duration']
for column in numeric_conversion:
    depression_file_cleaned[column] = pd.to_numeric(depression_file_cleaned[column], errors = 'coerce')

for column in numeric_conversion:
    median = depression_file_cleaned[column].median()
    depression_file_cleaned[column] = depression_file_cleaned[column].fillna(median)

categorical_columns = ['Gender', 'City', 'Degree', 'Profession', 'Dietary Habits']
depression_file_encoded = pd.get_dummies(depression_file_cleaned, columns=categorical_columns, drop_first=True)

print("Final Data Shape After Cleaning:", depression_file_encoded.shape)

categories = ['Age', 'CGPA', 'Work/Study Hours', 'Academic Pressure', 'Work Pressure', 'Study Satisfaction', 'Job Satisfaction', 'Financial Stress']
categories = [column for column in categories if column in depression_file_encoded.columns and depression_file_encoded[column].notna().sum() > 0]

In [ ]:
"""Heatmap of depression"""

plt.figure(figsize = (12, 10))
correlation = depression_file_encoded[categories].corr()
sns.heatmap(correlation, cmap = 'coolwarm')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
"""Countplot for depression stats"""

sns.countplot(x = 'Depression', data = depression_file)
plt.title('Depression Class Distribution')
plt.xticks([0, 1], ['No', 'Yes'])
plt.tight_layout()
plt.show()

In [ ]:
"""Histograms"""

for column in categories:
    plt.figure(figsize = (6, 4))
    sns.histplot(depression_file_encoded[column], kde = True)
    plt.title(f'Distribution of {column}')
    plt.xlabel(column)
    plt.tight_layout()
    plt.show()

In [ ]:
"""Boxplots of 'categories' vs 'depression'"""

for column in categories:
    plt.figure(figsize = (6, 4))
    sns.boxplot(x = 'Depression', y = column, data = depression_file)
    plt.title(f'{column} vs Depression')
    plt.xticks([0, 1], ['No', 'Yes'])
    plt.tight_layout()
    plt.show()

In [ ]:
"""Plots & Analysis with Debug Output"""

"""Gender vs Depression Plot""" #IT KEEPS SKIPPING DEPRESSSION PLOTS WHYYYYY - find where its dropping it!!!

print("\n=== Gender vs Depression ===")
if 'Gender' in depression_file.columns and depression_file['Depression'].notna().sum() > 0:
    print(depression_file[['Gender', 'Depression']].dropna().head())
    sns.countplot(x = 'Gender', hue = 'Depression', data = depression_file)
    plt.title('Depression Rate by Gender')
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: Required columns missing.")

"""Sleep Duration vs Depression"""

print("\n=== Sleep Duration vs Depression ===")
if all(column in depression_file.columns for column in ['Sleep Duration', 'Depression']):
    plot_depression_file = depression_file[['Sleep Duration', 'Depression']].dropna()
    print(f"Data shape: {plot_depression_file.shape}")
    print(plot_depression_file.head())
    if not plot_depression_file.empty:
        sns.boxplot(x = 'Depression', y = 'Sleep Duration', data = plot_depression_file)
        plt.title('Sleep Duration vs Depression')
        plt.xticks([0, 1], ['No', 'Yes'])
        plt.tight_layout()
        plt.show()
    else:
        print("Skipped: No data for plot")
else:
    print("Skipped: Required columns missing.")

"""Financial Stress by Degree"""

print("\n=== Financial Stress by Degree ===")
if 'Degree' in depression_file.columns and 'Financial Stress' in depression_file.columns:
    plot_depression_file = depression_file[['Degree', 'Financial Stress']].dropna()
    print(f"Data shape: {plot_depression_file.shape}")
    if not plot_depression_file.empty:
        plt.figure(figsize = (10, 5))
        sns.boxplot(x = 'Degree', y = 'Financial Stress', data = plot_depression_file)
        plt.xticks(rotation = 45)
        plt.title('Financial Stress by Degree')
        plt.tight_layout()
        plt.show()
    else:
        print("Skipped: No data for plot.")
else:
    print("Skipped: Required columns missing.")

"""CGPA vs Academic Pressure"""

print("\n=== CGPA vs Academic Pressure")
required_columns = ['CGPA', 'Academic Pressure']
if all(column in depression_file.columns for column in required_columns):
    plot_depression_file = depression_file[required_columns].dropna()
    print(f"Data shape: {plot_depression_file.shape}")
    print(plot_depression_file.head())
    if not plot_depression_file.empty:
        sns.jointplot(data = plot_depression_file, x = 'CGPA', y = 'Academic Pressure', kind = 'scatter')
        plt.tight_layout()
        plt.show()
    else:
        print("Skipped: No data for plot.")
else:
    print("Skipped: Required columns missing.")

"""Suicidal Thoughts vs Depression"""

print("\n=== Suicidal Thoughts vs Depression ===")
thoughts_col = 'Have you ever had suicidal thoughts ?'
if thoughts_col in depression_file.columns and depression_file['Depression'].notna().sum() > 0:
    plot_depression_file = depression_file[[thoughts_col, 'Depression']].dropna()
    print(f"Data shape: {plot_depression_file.shape}")
    print(plot_depression_file.head())
    if not plot_depression_file.empty:
        sns.countplot(x = thoughts_col, hue = 'Depression', data = plot_depression_file)
        plt.title('Suicidal Thoughts vs Depression')
        plt.xticks([0, 1], ['No', 'Yes'])
        plt.tight_layout()
        plt.show()
    else:
        print("Skipped: No data for plot.")
else:
    print("Skipped: Required columns missing.")
